In [0]:
# imports 

import requests
import uuid
from datetime import date, datetime, timedelta

In [0]:
# Constantes

MOEDAS = ["AUD", "CAD", "CHF", "DKK", "EUR", "GBP", "JPY", "NOK", "SEK", "USD"]

BASE_URL = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoMoedaPeriodo(moeda=@moeda,dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
)

In [0]:
# Funcao de request de uma moeda

def buscar_cotacao_moeda(moeda, data_inicial, data_final):
    params = {
        "@moeda": f"'{moeda}'",
        "@dataInicial": f"'{data_inicial}'",
        "@dataFinalCotacao": f"'{data_final}'",
        "$format": "json"
    }
    try:
        resposta = requests.get(BASE_URL, params=params, timeout=30)
        resposta.raise_for_status()
        dados = resposta.json().get("value", [])
        return {"moeda": moeda, "status": "ok", "dados": dados, "erro": None}
    except requests.exceptions.RequestException as e:
        return {"moeda": moeda, "status": "falha", "dados": None, "erro": str(e)}

In [0]:
# Campos de controle de execução 

run_id = str(uuid.uuid4())
insert_dt = datetime.utcnow().isoformat()


/home/spark-1fd21afd-1d26-4040-9e79-2c/.ipykernel/71/command-7393136512751318-832259509:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  insert_dt = datetime.utcnow().isoformat()


In [0]:
# Gap-detection
# Consulta a camada bronze para identificar datas faltantes

cobertura = spark.sql("""
    WITH cobertura AS (
        SELECT
            moeda,
            to_date(dataHoraCotacao) AS data_cotacao,
            COUNT(DISTINCT dataHoraCotacao) AS qtd_boletins
        FROM cambio_radar.bronze.cotacoes_ptax
        WHERE to_date(dataHoraCotacao) >= date_sub(current_date(), 30)
        GROUP BY moeda, to_date(dataHoraCotacao)
    )
    SELECT
        moeda,
        MIN(CASE WHEN qtd_boletins < 5 THEN data_cotacao END) AS primeira_data_incompleta,
        MAX(data_cotacao) AS ultima_data_com_dado
    FROM cobertura
    GROUP BY moeda
""").collect()

cobertura_por_moeda = {row["moeda"]: row for row in cobertura}

In [0]:
# Monta os pares pendentes

hoje = date.today()
pares_pendentes = []

for moeda in MOEDAS:
    if moeda not in cobertura_por_moeda:
        data_inicial = hoje - timedelta(days=30)
    else:
        registro = cobertura_por_moeda[moeda]
        if registro["primeira_data_incompleta"] is not None:
            data_inicial = registro["primeira_data_incompleta"]
        else:
            data_inicial = registro["ultima_data_com_dado"] + timedelta(days=1)

    pares_pendentes.append((moeda, data_inicial.strftime("%m-%d-%Y"), hoje.strftime("%m-%d-%Y")))

print(pares_pendentes)

[('AUD', '08-24-2026', '09-23-2026'), ('CAD', '08-24-2026', '09-23-2026'), ('CHF', '08-24-2026', '09-23-2026'), ('DKK', '08-24-2026', '09-23-2026'), ('EUR', '08-24-2026', '09-23-2026'), ('GBP', '08-24-2026', '09-23-2026'), ('JPY', '08-24-2026', '09-23-2026'), ('NOK', '08-24-2026', '09-23-2026'), ('SEK', '08-24-2026', '09-23-2026'), ('USD', '08-24-2026', '09-23-2026')]


In [0]:
# Loop para requests

resultados = []
dados_brutos = []

for moeda, data_inicial, data_final in pares_pendentes:
    resultado = buscar_cotacao_moeda(moeda, data_inicial, data_final)

    resultados.append({
        "moeda": resultado["moeda"],
        "status": resultado["status"],
        "erro": resultado["erro"]
    })

    if resultado["status"] == "ok":
        for registro in resultado["dados"]:
            registro["moeda"] = moeda
            registro["run_id"] = run_id
            registro["insert_dt"] = insert_dt
            dados_brutos.append(registro)

In [0]:
# Grava dados brutos na camada bronze

if dados_brutos:
    spark_df = spark.createDataFrame(dados_brutos)
    spark_df.write.format("delta").mode("append").saveAsTable("cambio_radar.bronze.cotacoes_ptax")
else:
    print("Nenhum dado retornado nesta execução — nada foi gravado na Bronze.")

In [0]:
# Check falhas
# Necessita Criar alertas e notificações

falhas = [r for r in resultados if r["status"] == "falha"]
if falhas:
    print("Falhas encontradas:", falhas)
else:
    print("Execução OK, sem falhas.")

Execução OK, sem falhas.


In [0]:
%sql

-- Validação Visual

SELECT * FROM cambio_radar.bronze.cotacoes_ptax ORDER BY moeda, dataHoraCotacao

cotacaoCompra,cotacaoVenda,dataHoraCotacao,insert_dt,moeda,paridadeCompra,paridadeVenda,run_id,tipoBoletim
3.6921,3.693,2026-08-24 10:10:12.94623,2026-09-23T01:51:09.198867,AUD,0.7162,0.7163,8d22892d-61ba-4e11-9717-5e7fe1bfe1d6,Abertura
3.6907,3.6921,2026-08-24 11:05:14.447056,2026-09-23T01:51:09.198867,AUD,0.7163,0.7165,8d22892d-61ba-4e11-9717-5e7fe1bfe1d6,Intermediário
3.6806,3.6831,2026-08-24 12:05:14.948605,2026-09-23T01:51:09.198867,AUD,0.7152,0.7156,8d22892d-61ba-4e11-9717-5e7fe1bfe1d6,Intermediário
3.6812,3.6821,2026-08-24 13:04:14.105669,2026-09-23T01:51:09.198867,AUD,0.715,0.7151,8d22892d-61ba-4e11-9717-5e7fe1bfe1d6,Intermediário
3.6827,3.6836,2026-08-24 13:04:14.216099,2026-09-23T01:51:09.198867,AUD,0.715,0.7151,8d22892d-61ba-4e11-9717-5e7fe1bfe1d6,Fechamento
3.686,3.687,2026-08-25 10:06:12.854962,2026-09-23T01:51:09.198867,AUD,0.7154,0.7155,8d22892d-61ba-4e11-9717-5e7fe1bfe1d6,Abertura
3.6845,3.686,2026-08-25 11:10:13.255402,2026-09-23T01:51:09.198867,AUD,0.7157,0.7159,8d22892d-61ba-4e11-9717-5e7fe1bfe1d6,Intermediário
3.6835,3.6844,2026-08-25 12:08:11.94469,2026-09-23T01:51:09.198867,AUD,0.7156,0.7157,8d22892d-61ba-4e11-9717-5e7fe1bfe1d6,Intermediário
3.6839,3.6848,2026-08-25 13:04:44.546109,2026-09-23T01:51:09.198867,AUD,0.7159,0.716,8d22892d-61ba-4e11-9717-5e7fe1bfe1d6,Intermediário
3.6857,3.6867,2026-08-25 13:04:44.743388,2026-09-23T01:51:09.198867,AUD,0.7159,0.716,8d22892d-61ba-4e11-9717-5e7fe1bfe1d6,Fechamento
